In [ ]:
import torch  # Core PyTorch library (tensors, autograd, operations)
import torchvision  # Provides datasets like FashionMNIST
import torchvision.transforms as transforms  # Used for preprocessing images


# -----------------------------
# Step 1: Load + Normalize
# -----------------------------

# Convert image to tensor AND normalize pixel values from [0,255] → [0,1]
# Transforms = preprocessing steps applied to data
transform = transforms.ToTensor()


# Load training dataset (60,000 images)
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',        # Folder where dataset will be stored
    train=True,           # Load training data
    download=True,        # Download if not already present
    transform=transform   # Apply preprocessing (tensor + normalization)
)

# Load test dataset (10,000 images)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',        # Same folder (reuses downloaded data)
    train=False,          # Load test data
    download=True,        # Download if not present
    transform=transform   # Apply same preprocessing
)

In [9]:
subset = torch.utils.data.Subset(train_dataset, range(5000))
train_loader = torch.utils.data.DataLoader(
    subset,
    batch_size=128,
    shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64
)


In [ ]:
# -----------------------------
# Step 3: Initialize parameters
# -----------------------------
# Input vector size:
# Each image: 28 × 28 → flattened → 784
input_size = 784

# Hidden layer sizes (number of neurons)
h1_size = 32
h2_size = 16

# Output layer size (number of classes)
output_size = 10

# -----------------------------
# WEIGHT MATRICES
# -----------------------------

# W1 maps input → hidden layer 1
# Shape: [784, 32]
W1 = torch.randn(input_size, h1_size, requires_grad=True)

# Bias for hidden layer 1Shape: [32]
b1 = torch.zeros(h1_size, requires_grad=True)


# W2 maps hidden1 → hidden2
# Shape: [32, 16]
W2 = torch.randn(h1_size, h2_size, requires_grad=True)

# Bias for hidden layer 2 [16]
b2 = torch.zeros(h2_size, requires_grad=True)


# W3 maps hidden2 → output
# Shape: [16, 10]
W3 = torch.randn(h2_size, output_size, requires_grad=True)

# Bias for output layer Shape: [10]
b3 = torch.zeros(output_size, requires_grad=True)


# -----------------------------
# HYPERPARAMETERS
# -----------------------------

# Learning rate: Controls update step size:
#   W = W - lr * dL/dW
lr = 0.01

# Number of passes over dataset
epochs = 5

In [ ]:
# -----------------------------
# Training loop
# -----------------------------
for epoch in range(epochs):
    total_loss = 0  # track loss per epoch

    for images, labels in train_loader:

        # ---- Flatten images ----
        # [batch,1,28,28] → [batch,784]
        x = images.reshape(-1, 784)

        # ---- Forward pass ----
        # Layer 1
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        # Layer 2
        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        # Output layer (logits)
        logits = h2 @ W3 + b3

        # ---- Loss ----
        # CrossEntropy = softmax + log + compare with labels
        loss = torch.nn.functional.cross_entropy(logits, labels)

        # ---- Backward pass ----
        loss.backward()

        # ---- Update parameters ----
        with torch.no_grad():
            W1 -= lr * W1.grad
            b1 -= lr * b1.grad

            W2 -= lr * W2.grad
            b2 -= lr * b2.grad

            W3 -= lr * W3.grad
            b3 -= lr * b3.grad

        # ---- Reset gradients ----
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()
        W3.grad.zero_()
        b3.grad.zero_()

        # Accumulate loss
        total_loss += loss.item()

    # Print epoch loss
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 96.1059
Epoch 2, Loss: 90.3339
Epoch 3, Loss: 86.7172


In [6]:
# -----------------------------
# Accuracy (Evaluation)
# -----------------------------

correct = 0  # count correct predictions
total = 0    # total samples

# Disable gradient computation (faster + no memory overhead)
with torch.no_grad():

    for images, labels in test_loader:

        # Flatten images: [batch,1,28,28] → [batch,784]
        x = images.reshape(-1, 784)

        # Forward pass (same as training)
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        # Get predicted class (index of max score)
        predictions = torch.argmax(logits, dim=1)

        # Count correct predictions
        correct += (predictions == labels).sum().item()

        # Count total samples
        total += labels.size(0)

# Compute accuracy
accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 30.13%


In [10]:
# General Idea 

# You built a Fully Connected Neural Network (FCNN) to classify images from the FashionMNIST dataset.
# Each image is something like a shirt, shoe, bag, etc., and your model is trying to answer:
# “Given this image, which of the 10 categories does it belong to?

# Image → Numbers → Neural Network → Prediction → Compare with truth → Improve
# Step 1: Data comes in (Images → Numbers)
# Original images are 28 × 28 pixels
# Each pixel = intensity value (0–255)

# You convert them into:
# Tensor (numerical format)
# Normalize to 0–1 range

#  Why?
# Because neural networks only understand numbers, not images.

# Each image:
# 28 × 28 → 784
# So instead of thinking “image”, your model sees:
# [0.1, 0.7, 0.0, ..., 0.3]   (784 values)

# Brutal truth:
# You are destroying spatial structure here.
# This is why FCNN is weaker than CNNs for images. But for labs, it's fine.

# Step 3: The Neural Network (Core idea)
# You created a 3-layer transformation pipeline:
# Input (784)
#    ↓
# Hidden Layer 1 (32 neurons)
#    ↓
# Hidden Layer 2 (16 neurons)
#    ↓
# Output Layer (10 classes)


# What each layer does (intuitively)
# Each layer:
# Output = (Input × Weights) + Bias → Activation
# So what’s happening?
# Weights = what the model learns (importance of each input)
# Bias = adjustment term
# ReLU = introduces non-linearity (otherwise useless model)

# How to visualize learning
# Layer 1 → learns basic patterns
# Layer 2 → combines them into higher-level patterns
# Output → decides which class
# Even though this is not a CNN, it still tries to learn patterns statistically.

# Step 4: Output = logits
# Final output = 10 numbers
# Example:
# [2.1, -1.2, 0.5, 4.8, ...]
# These are scores (logits), not probabilities.

#  Step 5: Loss (How wrong are you?)
# You use:
# Cross Entropy Loss
# This does:

# Convert logits → probabilities (softmax)
# Compare with actual label
# Penalize wrong predictions
# Lower loss = better predictions

# Step 6: Backpropagation (How to improve)
# This is the core magic:

# PyTorch calculates:
# “How much did each weight contribute to the error?”
# That gives:
# gradients = ∂Loss / ∂Weight

# Step 7: Update weights
# You manually do:
# W = W - learning_rate × gradient

#  Meaning:
# If a weight caused error → reduce it
# If helpful → adjust accordingly

# This is gradient descent

# Step 8: Repeat (Epochs)
# You loop over data multiple times:
# Each pass = epoch
# Each batch = mini chunk of data

# Over time:
# Loss ↓
# Accuracy ↑


# What your model is learning (real intuition)
# It is NOT “seeing clothes”
# It is learning:
# pixel intensity patterns
# statistical correlations

# Example:
# certain pixel combinations → “shoe”
# others → “shirt”

In [11]:
# Cell 2
# What problem this block solves

# You don’t feed the entire dataset at once. Instead, you:
# Dataset → Small chunks (batches) → Model
# Why?

# Memory constraints (RAM/GPU)
# Faster + more stable learning
# Better generalization 

# 2. DataLoader — turning data into batches
# Training loader
# train_loader = torch.utils.data.DataLoader(
#     subset,
#     batch_size=128,
#     shuffle=True
# )

# What it does:

# It converts your dataset into:

# [128 images] → batch 1
# [128 images] → batch 2

# What happens internally

# Each iteration gives you:
# images, labels
# Where:

# images → shape: [128, 1, 28, 28]
# labels → shape: [128]
# Key parameter: batch_size=128
# You process 128 images at once
# Trade-off:

#  shuffle=True
# This is critical.
# Each epoch:  Data order is randomized

# Why?
# If you don’t shuffle:
# Model sees same pattern order every time → poor learning
# With shuffle:
# Better generalization

# Big picture (connect everything)

# Your training loop:

# for images, labels in train_loader:

# means:

# Step 1: Take 128 images
# Step 2: Run model
# Step 3: Update weights
# Step 4: Take next 128 images

# Important intuition

# Without DataLoader, you'd be doing:

# for each image:
#     train model

# Which is:
# VERY slow
# VERY noisy

In [13]:
# Cell 3
# What this means (real intuition)
# 784 → raw pixel input
# 32 → compress information (first abstraction)
# 16 → compress further (higher abstraction)
# 10 → final decision (one per class)

#  2. Weights = what the model learns
#  W1 (Input → Hidden 1)
# W1 = torch.randn(784, 32)
# Think:
# 784 inputs → 32 neurons
# Each neuron:
# “looks at all 784 pixels and decides something”

# So:
# Each column of W1 = one neuron
# Each neuron has 784 weights

#  W2 (Hidden1 → Hidden2)
# W2 = torch.randn(32, 16)

# Now:

# 32 features → 16 refined features

#  You’re no longer dealing with pixels
#  You’re combining patterns of patterns

# 🔹 W3 (Hidden2 → Output)
# W3 = torch.randn(16, 10)
# Final mapping:
# 16 features → 10 class scores
# Each output neuron = one class (shoe, shirt, etc.)

# b1 = torch.zeros(32)

# Bias lets neuron say:

# “Even if input is zero, I can still activate”
# First layer:
# X:        (batch × 784)
# W1:       (784 × 32)
# Result:   (batch × 32)

#  Why this works:
# (784 matches 784) → valid matrix multiplication

In [ ]:
# Cell 4
# Each batch goes through:
# Input → Predict → Measure error → Fix parameters → Repeat
# That’s it. The rest is just mechanics.